# 01 평가 데이터의 기준과 분리

목표: 답변 가능 여부·필수 근거를 검증하고 같은 의도 표현이 dev/test에 섞이는 문제를 재현합니다.

**준비:** 저장소 터미널에서 python -m pip install -r requirements/week11-offline.txt 실행.
VS Code 커널에서 저장소 .venv의 **Python 3.11** 선택 후 위에서 아래로 실행합니다.
각 파일은 다른 Notebook의 변수를 요구하지 않습니다. GPU·Ollama·API 키는 필요 없습니다.

**데이터:** 교수 작성 가상 쇼핑 정책입니다. 실제 기업 규정이나 고객 기록이 아닙니다.
핵심 단계는 수업에서 함께 실행하고 확장 실험은 복습용입니다. [실습 안내](../README.md).


In [ ]:
from pathlib import Path
import sys, json

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week11/lab/evaluation.py").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("저장소 안에서 Notebook을 실행하세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("저장소 .venv의 Python 3.11 커널을 선택하세요.")
import pandas as pd
from IPython.display import display
from week11.lab.evaluation import load_data, validate_data

docs, questions = load_data()
dev = [q for q in questions if q["split"] == "dev"]
test = [q for q in questions if q["split"] == "test"]
print("환경 준비 완료", sys.version.split()[0], "문서", len(docs), "질문", len(questions))

## 1. 현재 문서와 과거 문서

폐기된 정책은 정답 근거가 아닙니다. 메타데이터로 검색 전에 필터링합니다.

In [ ]:
display(pd.DataFrame(docs)[["id", "title", "status", "version"]])

**확인과 해석**

현재 18개, 과거 4개입니다. 과거 문서는 실패를 보여주기 위해 남겨 두었습니다.

## 2. 평가 항목 구조

query는 입력, gold_ids는 필수 문서 집합, reference는 사람이 확인할 의미 기준입니다.

In [ ]:
display(pd.DataFrame(questions).head(6))
print(json.dumps(questions[4], ensure_ascii=False, indent=2))

**확인과 해석**

교환 기간과 비용은 두 문서가 필요합니다. 정답을 검색 입력이나 색인에 섞지 마세요.

## 3. 문항과 독립 의도

말투 두 개가 곧 독립적인 두 문제는 아닙니다.

In [ ]:
df = pd.DataFrame(questions)
display(pd.crosstab([df["split"], df["slice"]], df["style"]))
display(df.groupby("split").agg(questions=("id", "count"), groups=("group", "nunique")))

**확인과 해석**

각 split은 24문항·12개 의도입니다. test 복수 근거 의도는 하나뿐이므로 일반화에 주의합니다.

## 4. 정답 무결성

삭제된 문서나 과거 문서를 정답으로 쓰는 오류를 차단합니다.

In [ ]:
import copy

bad = copy.deepcopy(questions)
bad[0]["gold_ids"] = ["D999"]
try:
    validate_data(docs, bad)
except ValueError as error:
    print("예상된 실패:", error)
validate_data(docs, questions)

**확인과 해석**

데이터 오류를 조용히 0점으로 만들지 않습니다.

## 5. 표현 변형 누수

같은 의도 그룹은 한 split에만 둡니다.

In [ ]:
bad = copy.deepcopy(questions)
bad[1]["split"] = "test"
try:
    validate_data(docs, bad)
except ValueError as error:
    print("누수 차단:", error)
assert {q["group"] for q in dev}.isdisjoint({q["group"] for q in test})

**확인과 해석**

공유 정책 문서를 검색하는 조건이며 신규 도메인 일반화 실험은 아닙니다.

## 6. 합성 질문 검토

자동 생성은 정답 검수를 대체하지 못합니다. 아래는 사람이 쓴 검토 예시입니다.

In [ ]:
display(
    pd.DataFrame(
        [
            {"query": "제주 추가 배송비는?", "decision": "채택 후보"},
            {"query": "무료 반품은 언제까지?", "decision": "무료라는 잘못된 전제 수정"},
            {"query": "내 운동화 내일 와요?", "decision": "답변 불가 사례"},
        ]
    )
)

**확인과 해석**

잘못된 전제·문서 밖 정보·개별 주문 상태를 구분합니다.

## 확장 실험과 프로젝트 적용

1. 단일·복수 근거·답변 불가 질문을 하나씩 작성합니다.
2. group·출처·검토 상태를 기록합니다.
3. 공개 test를 반복 최적화한 뒤에는 새 평가 세트를 준비합니다.

**실행 실패 시:** ModuleNotFoundError는 커널과 설치 터미널이 같은 .venv인지 확인합니다.
ROOT 오류는 저장소 안에서 파일을 열면 해결됩니다. 변수가 섞였다면 Restart Kernel 후 Run All.
예외 실험은 try/except로 설명하므로 정상 실행에서는 빨간 오류로 끝나지 않습니다.

**기록:** 변경 설정 / 예상 / 실제 관찰 / 실패 질문 ID / 다음 행동을 5줄로 적습니다.
개인 프로젝트 원문·연락처·API 키는 공개 GitHub에 올리지 않습니다.
